# Library

In [15]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

# Data Loading

In [16]:
df = pd.read_csv('laptop_price.csv', encoding='latin-1')
df

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_euros
0,1,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37kg,1339.69
1,2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
2,3,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,No OS,1.86kg,575.00
3,4,Apple,MacBook Pro,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16GB,512GB SSD,AMD Radeon Pro 455,macOS,1.83kg,2537.45
4,5,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8GB,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37kg,1803.60
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1298,1316,Lenovo,Yoga 500-14ISK,2 in 1 Convertible,14.0,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i7 6500U 2.5GHz,4GB,128GB SSD,Intel HD Graphics 520,Windows 10,1.8kg,638.00
1299,1317,Lenovo,Yoga 900-13ISK,2 in 1 Convertible,13.3,IPS Panel Quad HD+ / Touchscreen 3200x1800,Intel Core i7 6500U 2.5GHz,16GB,512GB SSD,Intel HD Graphics 520,Windows 10,1.3kg,1499.00
1300,1318,Lenovo,IdeaPad 100S-14IBR,Notebook,14.0,1366x768,Intel Celeron Dual Core N3050 1.6GHz,2GB,64GB Flash Storage,Intel HD Graphics,Windows 10,1.5kg,229.00
1301,1319,HP,15-AC110nv (i7-6500U/6GB/1TB/Radeon,Notebook,15.6,1366x768,Intel Core i7 6500U 2.5GHz,6GB,1TB HDD,AMD Radeon R5 M330,Windows 10,2.19kg,764.00


In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1303 entries, 0 to 1302
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   laptop_ID         1303 non-null   int64  
 1   Company           1303 non-null   object 
 2   Product           1303 non-null   object 
 3   TypeName          1303 non-null   object 
 4   Inches            1303 non-null   float64
 5   ScreenResolution  1303 non-null   object 
 6   Cpu               1303 non-null   object 
 7   Ram               1303 non-null   object 
 8   Memory            1303 non-null   object 
 9   Gpu               1303 non-null   object 
 10  OpSys             1303 non-null   object 
 11  Weight            1303 non-null   object 
 12  Price_euros       1303 non-null   float64
dtypes: float64(2), int64(1), object(10)
memory usage: 132.5+ KB


# Feature Engineering

## Vectorization of categorical columns

#### RAM

In [18]:
df['Ram_GB'] = df['Ram'].str.replace('GB','').astype(int)

#### Weight

In [19]:
df['Weight_kg'] = df['Weight'].str.replace('kg','').astype(float)

#### Memory

In [20]:
def memory_parse(memory_str):
    total = 0
    parts = memory_str.split('+')

    for part in parts:
        part = part.strip()

        if 'TB' in part:
            num = float(part.split('TB')[0])
            total += num*1024 #Convert ke GB

        elif 'GB' in part:
            num = float(part.split('GB')[0])
            total += num

    return total

df['Storage_GB'] = df['Memory'].apply(memory_parse)

### GPU

In [21]:
def get_gpu_brand(gpu):
    if 'Nvidia' in gpu :
        return 'Nvidia'
    elif 'AMD' in gpu:
        return 'AMD'
    else:
        return 'Intel'

df['Gpu_Brand'] = df['Gpu'].apply(get_gpu_brand)

#### Screen Resolution

In [22]:
def get_resolution(res_str):
    if '3840' in res_str:
        return '4K'
    elif '3200' in res_str or '2880' in res_str or '2736' in res_str:
        return 'QHD'
    elif '2560' in res_str or '2304' in res_str or '2400' in res_str or '2256' in res_str or '2160' in res_str:
        return 'FHD+'
    elif '1920' in res_str or '1600' in res_str:
        return 'FHD'
    else:
        return 'HD'
    
df['Resolution'] = df['ScreenResolution'].apply(get_resolution)

#### CPU

In [23]:
def get_cpu_brand(cpu_str):
    if 'Intel' in cpu_str:
        return 'Intel'
    elif 'AMD' in cpu_str:
        return 'AMD'
    else:
        return 'Other'
    
def get_cpu_tier(cpu_str):
    if 'i7' in cpu_str or 'i9' in cpu_str or 'Ryzen 7' in cpu_str or 'Ryzen 9' in cpu_str:
        return 'High'
    elif 'i5' in cpu_str or 'Ryzen 5' in cpu_str:
        return 'Mid'
    elif 'i3' in cpu_str or 'Ryzen 3' in cpu_str:
        return 'Entry'
    else:
        return 'Other'

df['Cpu_Brand'] = df['Cpu'].apply(get_cpu_brand)
df['Cpu_Tier']  = df['Cpu'].apply(get_cpu_tier)

#### Touchscreen

In [24]:
df['Touchscreen'] = df['ScreenResolution'].apply(lambda x: 1 if 'Touchscreen' in x else 0)

In [26]:
df[['Memory', 'Storage_GB', 'Ram', 'Ram_GB', 'Weight', 'Weight_kg', 'Gpu', 'Gpu_Brand', 'ScreenResolution','Resolution', 'Touchscreen', 'Cpu', 'Cpu_Brand', 'Cpu_Tier']].head(25)

,Memory,Storage_GB,Ram,Ram_GB,Weight,Weight_kg,Gpu,Gpu_Brand,ScreenResolution,Resolution,Touchscreen,Cpu,Cpu_Brand,Cpu_Tier
0,128GB SSD,128.0,8GB,8,1.37kg,1.37,Intel Iris Plus Graphics 640,Intel,IPS Panel Retina Display 2560x1600,FHD+,0,Intel Core i5 2.3GHz,Intel,Mid
1,128GB Flash Storage,128.0,8GB,8,1.34kg,1.34,Intel HD Graphics 6000,Intel,1440x900,HD,0,Intel Core i5 1.8GHz,Intel,Mid
2,256GB SSD,256.0,8GB,8,1.86kg,1.86,Intel HD Graphics 620,Intel,Full HD 1920x1080,FHD,0,Intel Core i5 7200U 2.5GHz,Intel,Mid
3,512GB SSD,512.0,16GB,16,1.83kg,1.83,AMD Radeon Pro 455,AMD,IPS Panel Retina Display 2880x1800,QHD,0,Intel Core i7 2.7GHz,Intel,High
4,256GB SSD,256.0,8GB,8,1.37kg,1.37,Intel Iris Plus Graphics 650,Intel,IPS Panel Retina Display 2560x1600,FHD+,0,Intel Core i5 3.1GHz,Intel,Mid
5,500GB HDD,500.0,4GB,4,2.1kg,2.10,AMD Radeon R5,AMD,1366x768,HD,0,AMD A9-Series 9420 3GHz,AMD,Other
6,256GB Flash Storage,256.0,16GB,16,2.04kg,2.04,Intel Iris Pro Graphics,Intel,IPS Panel Retina Display 2880x1800,QHD,0,Intel Core i7 2.2GHz,Intel,High
7,256GB Flash Storage,256.0,8GB,8,1.34kg,1.34,Intel HD Graphics 6000,Intel,1440x900,HD,0,Intel Core i5 1.8GHz,Intel,Mid
8,512GB SSD,512.0,16GB,16,1.3kg,1.30,Nvidia GeForce MX150,Nvidia,Full HD 1920x1080,FHD,0,Intel Core i7 8550U 1.8GHz,Intel,High
9,256GB SSD,256.0,8GB,8,1.6kg,1.60,Intel UHD Graphics 620,Intel,IPS Panel Full HD 1920x1080,FHD,0,Intel Core i5 8250U 1.6GHz,Intel,Mid


#### Feature Scaling

In [ ]:
scaler = MinMaxScaler()

df[['Norm_Ram', 'Norm_Storage', 'Norm_Weight', 'Norm_Price']] = scaler.fit_transform(df[['Ram_GB', 'Storage_GB', 'Weight_kg', 'Price_euros']])

#### Encode


In [ ]:
type_encode = pd.get_dummies(df['TypeName'],prefix='type')
gpu_encode = pd.get_dummies(df['Gpu_Brand'],prefix='gpu')
resolution_encode = pd.get_dummies(df['Resolution'], prefix='res')
cpu_brand_encode = pd.get_dummies(df['Cpu_Brand'],  prefix='cpu')
cpu_tier_encode   = pd.get_dummies(df['Cpu_Tier'],   prefix='tier')

## Vector

In [ ]:
laptop_vector = pd.concat([df[['Norm_Ram', 'Norm_Storage', 'Norm_Weight', 'Norm_Price', 'Touchscreen']],
                           type_encode, 
                           gpu_encode,
                           resolution_encode,
                           cpu_brand_encode,
                           cpu_tier_encode
                           ], axis=1)

laptop_vector.index = df['Product']

In [ ]:
laptop_vector

,Norm_Ram,Norm_Storage,Norm_Weight,Norm_Price,Touchscreen,type_2 in 1 Convertible,type_Gaming,type_Netbook,type_Notebook,type_Ultrabook,...,res_FHD+,res_HD,res_QHD,cpu_AMD,cpu_Intel,cpu_Other,tier_Entry,tier_High,tier_Mid,tier_Other
Product,,,,,,,,,,,,,,,,,,,,,
MacBook Pro,0.096774,0.047022,0.169576,0.196741,0,False,False,False,False,True,...,True,False,False,False,True,False,False,False,True,False
Macbook Air,0.096774,0.047022,0.162095,0.122353,0,False,False,False,False,True,...,False,True,False,False,True,False,False,False,True,False
250 G6,0.096774,0.097179,0.291771,0.067679,0,False,False,False,True,False,...,False,False,False,False,True,False,False,False,True,False
MacBook Pro,0.225806,0.197492,0.284289,0.398895,0,False,False,False,False,True,...,False,False,True,False,True,False,False,True,False,False
MacBook Pro,0.096774,0.097179,0.169576,0.275038,0,False,False,False,False,True,...,True,False,False,False,True,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Yoga 500-14ISK,0.032258,0.047022,0.276808,0.078312,1,True,False,False,False,False,...,False,False,False,False,True,False,False,True,False,False
Yoga 900-13ISK,0.225806,0.197492,0.152120,0.223629,1,True,False,False,False,False,...,False,False,True,False,True,False,False,True,False,False
IdeaPad 100S-14IBR,0.000000,0.021944,0.201995,0.009283,0,False,False,False,True,False,...,False,True,False,False,True,False,False,False,False,True


### Cosine similarity

In [ ]:
def cosine_sim(vect1,vect2):
  norm_1 = np.linalg.norm(vect1)
  norm_2 = np.linalg.norm(vect2)

  cos_sim = (vect1 @ vect2) / (norm_1 * norm_2)
  return cos_sim


### Recommender System

In [ ]:
def recsys(type_name, gpu_brand, ram_user, storage_user, weight_user, price_user, resolution, touchscreen, cpu_brand, cpu_tier, top_N=5):
    
    # Input User
    user_input = pd.DataFrame([[ram_user, storage_user, weight_user, price_user]],
                              columns=['Ram_GB', 'Storage_GB', 'Weight_kg', 'Price_euros'])
    user_norm = scaler.transform(user_input)[0]

    # Query vector

    cossim = pd.Series(0.0,index=laptop_vector.columns)
    cossim[f'type_{type_name}'] = 1.0
    cossim[f'gpu_{gpu_brand}'] = 1.0
    cossim[f'res_{resolution}'] = 1.0    
    cossim[f'cpu_{cpu_brand}'] = 1.0
    cossim[f'tier_{cpu_tier}'] = 1.0
    cossim['Touchscreen']   = touchscreen
    cossim['Norm_Ram']      = user_norm[0]
    cossim['Norm_Storage']  = user_norm[1]
    cossim['Norm_Weight']   = user_norm[2]
    cossim['Norm_Price']    = user_norm[3]

    # Cosine similarity calculation

    score = [cosine_sim(cossim.values, x) for x in laptop_vector.values]
    result = pd.Series(score, index=laptop_vector.index)
    top_result = result.sort_values(ascending=False).head(top_N)

    return top_result

### Model Inference

Mencoba testing reccomender system dengan kriteria
- type_name = Gaming
- gpu_brand = Intel
- ram_gb = 32
- storage_gb = 512
- weight_kg = 1.5
- price_euros = 1200
- resolution= 'FHD',
- touchscreen= 1,
- cpu_brand= 'Intel',
- cpu_tier= 'High'

In [ ]:
recsys(
    type_name= 'Gaming',
    gpu_brand= 'Nvidia',
    ram_user= 32,
    storage_user= 512,
    weight_user= 1.5,
    price_user= 1200,
    resolution= 'FHD',
    touchscreen= 1,
    cpu_brand= 'Intel',
    cpu_tier= 'High'
)

Product
IdeaPad Y700-15ISK    0.984067
Rog G752VL-UH71T      0.957460
Legion Y520-15IKBN    0.907729
Legion Y520-15IKBN    0.907641
ROG Zephyrus          0.906161
dtype: float64